In [1]:
import pandas as pd
from functions.pred import *
from functions.xai import *
from functions.eval import *
from tqdm.notebook import tqdm
tqdm.pandas()

In [2]:
model_pred = pd.read_csv("data/model_pred/model_pred.csv")

In [3]:
model_pred_col = "Camelbert-MSA"
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
# model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
# model_pred_col = "AraBert"

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device)

Device set to use cuda


In [ ]:
lime_num_samples = 100
shap_max_evals = 100
ig_n_steps = 50

In [6]:
lime_explainer = LimeExplainer(model_name, device, num_samples=lime_num_samples)
shap_explainer = ShapExplainer(model_name, device, max_evals=shap_max_evals)
ig_explainer = IgExplainer(model_name, device, n_steps=ig_n_steps)
dl_explainer = DeepLiftExplainer(model_name, device)
ensemble_explainer = EnsembleExplainer(mean=True, median=True)

In [7]:
text = "هذا الفيلم كان رائعًا وممتعًا للغاية!"
label = "positive"
# text = "لم يعجبني هذا الفيلم على الإطلاق، كان مملاً وسيئًا."
# label = "negative"

In [8]:
lime_res = lime_explainer.explain(text, label)
lime_res

[[0, 'هذا', -1.0],
 [1, 'الفيلم', -0.2276370014668555],
 [2, 'كان', 0.12501485309969063],
 [3, 'رائعا', 1.0],
 [4, 'وممت', 0.42999305610691674],
 [5, '##عا', -0.4942492594059713],
 [6, 'للغاية', -0.0977992903729491],
 [7, '!', -0.6126524600959604]]

In [9]:
shap_res = shap_explainer.explain(text, label)
shap_res

PartitionExplainer explainer: 2it [00:16, 16.09s/it]               


[[0, 'هذا', -0.92789547470662],
 [1, 'الفيلم', -1.0],
 [2, 'كان', -0.969333497739321],
 [3, 'رائعا', 1.0],
 [4, 'وممت', 0.7901814538622929],
 [5, '##عا', -0.7032982933765937],
 [6, 'للغاية', 0.7113569211358037],
 [7, '!', -0.29408616634479157]]

In [10]:
ig_res = ig_explainer.explain(text, label)
ig_res

[[0, 'هذا', np.float64(-0.7181474037987122)],
 [1, 'الفيلم', np.float64(-1.0)],
 [2, 'كان', np.float64(-0.1690194824387785)],
 [3, 'رائعا', np.float64(1.0)],
 [4, 'وممت', np.float64(-0.18451843525043776)],
 [5, '##عا', np.float64(-0.13060825004829013)],
 [6, 'للغاية', np.float64(0.004056138674615184)],
 [7, '!', np.float64(-0.24684846423196616)]]

In [11]:
dl_res = dl_explainer.explain(text, label)
dl_res

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\_utils\gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\attr\_core\deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  warnings.warn(


[[0, 'هذا', -1.0],
 [1, 'الفيلم', 0.36039090156555176],
 [2, 'كان', 0.3185434341430664],
 [3, 'رائعا', -0.12182152271270752],
 [4, 'وممت', -0.1745615005493164],
 [5, '##عا', 0.22616827487945557],
 [6, 'للغاية', -0.5793724656105042],
 [7, '!', 1.0]]

In [12]:
ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res, ig_results=ig_res, dl_results=dl_res)

([[0, 'هذا', np.float64(-0.9115107196263331)],
  [1, 'الفيلم', np.float64(-0.46681152497532596)],
  [2, 'كان', np.float64(-0.1736986732338356)],
  [3, 'رائعا', np.float64(0.7195446193218231)],
  [4, 'وممت', np.float64(0.2152736435423639)],
  [5, '##عا', np.float64(-0.2754968819878499)],
  [6, 'للغاية', np.float64(0.009560325956741417)],
  [7, '!', np.float64(-0.038396772668179535)]],
 [[0, 'هذا', np.float64(-0.9639477373533101)],
  [1, 'الفيلم', np.float64(-0.6138185007334278)],
  [2, 'كان', np.float64(-0.022002314669543943)],
  [3, 'رائعا', np.float64(1.0)],
  [4, 'وممت', np.float64(0.12771577777880017)],
  [5, '##عا', np.float64(-0.3124287547271307)],
  [6, 'للغاية', np.float64(-0.04687157584916696)],
  [7, '!', np.float64(-0.27046731528837886)]])

In [13]:
xai_exec = model_pred[["text", model_pred_col]]
xai_exec

,text,Camelbert-MSA
0,: وصلنا لاقتصاد اسوء من سوريا والعراق ومن غير ...,negative
1,كاني ويست، دريك، نيكي، بيونسيه، قاقا,neutral
2,على فكره شركة محترمه حداعطوني كيبل كهديه ويوم ...,positive
3,: المتعه افضل من الزواج لهتك اعراض عامة #الشيع...,negative
4,القوات البرية السعودية والقوات الفرنسية الخاصة...,neutral
...,...,...
3028,قال رسول الله ﷺ(إذا سمعتم الطاعون بأرض، فلا تد...,neutral
3029,: ماركا | لم ينهزم ريال مدريد في أخر 23 مباراة...,positive
3030,: #مليارات_العمره_على_هوى_مصر سقوط بشار الكلب ...,negative
3031,حد معاه ويندوز 10 ؟,neutral


In [ ]:
xai_exec["LIME"] = xai_exec.progress_apply(lambda x: lime_explainer.explain(x["text"], x[model_pred_col]), axis=1)

In [ ]:
SHAP_BATCH_SIZE = 256
shap_results = shap_explainer.explain_batch(
    xai_exec["text"].tolist(),
    xai_exec[model_pred_col].tolist(),
    shap_batch_size=SHAP_BATCH_SIZE,
)
xai_exec["SHAP"] = shap_results

PartitionExplainer explainer: 3034it [04:51, 10.30it/s]                          


In [ ]:
xai_exec["IG"] = xai_exec.progress_apply(lambda x: ig_explainer.explain(x["text"], x[model_pred_col]), axis=1)

In [ ]:
xai_exec["DeepLIFT"] = xai_exec.progress_apply(lambda x: dl_explainer.explain(x["text"], x[model_pred_col]), axis=1)

In [ ]:
xai_exec[["EnsembleXAI_LIME_SHAP_IG_DL_mean", "EnsembleXAI_LIME_SHAP_IG_DL_median"]] = \
    xai_exec.progress_apply(lambda x: ensemble_explainer.explain(lime_results=x["LIME"], 
            shap_results=x["SHAP"], ig_results=x["IG"], dl_results=x["DeepLIFT"]),
        axis=1, result_type="expand")

  0%|          | 0/3033 [00:00<?, ?it/s]

In [ ]:
xai_exec[["EnsembleXAI_LIME_SHAP_IG_mean", "EnsembleXAI_LIME_SHAP_IG_median"]] = \
    xai_exec.progress_apply(lambda x: ensemble_explainer.explain(lime_results=x["LIME"], 
            shap_results=x["SHAP"], ig_results=x["IG"]), axis=1, result_type="expand")

  0%|          | 0/3033 [00:00<?, ?it/s]

In [ ]:
xai_exec[["EnsembleXAI_LIME_SHAP_mean", "EnsembleXAI_LIME_SHAP_median"]] = \
    xai_exec.progress_apply(lambda x: ensemble_explainer.explain(lime_results=x["LIME"], 
            shap_results=x["SHAP"]), axis=1, result_type="expand")

  0%|          | 0/3033 [00:00<?, ?it/s]

In [60]:
xai_exec.to_csv("data/xai_exec/xai_exec_" + model_pred_col + ".csv", index=False)